# 主观评测 · 双维度 MOS 1–5

读 `v1112_compare/attr.json`，**按片段分组**：同一个片段的 5 个方法结果（ours / avtr-1 / avatar forcing / ovi / GT）在页面上并排同时播放，
评价者对每一个分别打两个维度的分（1 很差 … 5 很好）：

| 维度 | 含义 |
| --- | --- |
| 反应自然程度 | 面部/口型/肢体的反应是否自然，有没有明显崩坏、模糊、伪影 |
| 反应时机恰当程度 | 反应出现的时机是否合理，与对方说话 / 动作是否同步 |

- 组内 5 个视频的位置**每轮随机打乱**，只用「视频 A / B / C / D / E」标识，
  不显示方法名、文件名和片段编号；方法归属只写进 CSV。
- 每次提交落盘到 `v1112_compare/logs/YYYYMMDD_HHMMSS.csv`，一行一个视频，
  列：`group_index / slot / order_index / clip_id / method / naturalness / timing / avg / subject / session_id / seed / rated_at / path`；
  同目录另写 `YYYYMMDD_HHMMSS_summary.csv` 存方法级均值。
- 一组 5 个一起打分，不用来回翻页；「本组评完自动下一组」只在整组刚填满时触发，
  事后回去改分不会被弹走。全部组评完才能提交。
- **评分只在内存里**：提交前刷新页面会丢。

运行：`conda activate volia_viewer && jupyter notebook user_study.ipynb`，
或用 `voila user_study.ipynb` 起一个只有评测界面的页面给评价者。
最后一个代码格是多人评测结束后的汇总统计，读 `logs/` 下所有会话 CSV。


In [ ]:
import os
import csv
import json
import glob
import base64
import random
import datetime
import mimetypes
import statistics
from collections import OrderedDict, defaultdict

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output, Javascript

# ============================== 配置 ==============================
VIDEO_MAX_WIDTH = 280         # 单个视频正方形边长上限；5 格并排要窄一点，704x704 等比缩小不裁切
GROUP_COLUMNS = 5             # 每行最多并排几个（attr.json 里一个片段 5 个方法，正好一行）
SCORE_MIN, SCORE_MAX = 1, 5   # MOS 量程
AUTO_ADVANCE_DEFAULT = True   # 整组刚评完时自动跳到下一组
WRITE_SUMMARY_CSV = True      # 额外写一份 <stamp>_summary.csv（方法级均值）
SHOW_SUMMARY_ON_SUBMIT = True # 交卷后展示方法均值（此时盲测已结束）

SCORE_LABELS = {1: "很差", 2: "较差", 3: "一般", 4: "较好", 5: "很好"}

# 两个评分维度：(键, 显示名, 悬停说明)
DIMENSIONS = (
    ("naturalness", "自然程度", "反应的面部/口型/肢体是否自然，有无崩坏、模糊、伪影"),
    ("timing", "时机恰当", "反应的时机是否合理，与对方的说话/动作是否同步"),
)
DIM_KEYS = tuple(d[0] for d in DIMENSIONS)
DIM_NAMES = {d[0]: d[1] for d in DIMENSIONS}

# 字段顺序即 CSV 列顺序，改这里不影响历史文件
CSV_FIELDS = ["group_index", "slot", "order_index", "clip_id", "method",
              "naturalness", "timing", "avg", "subject", "session_id",
              "seed", "rated_at", "path"]


def find_project_root():
    """从 cwd 向上找含 v1112_compare/attr.json 的目录，找不到就回落到固定路径。

    这样在项目根目录、v1112_compare/ 之内、或别处启动内核都能定位到同一份 attr.json。
    """
    here = os.path.abspath(os.getcwd())
    for _ in range(6):
        if os.path.isfile(os.path.join(here, "v1112_compare", "attr.json")):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    return "/ytech_milm_intern/chenhejia/projects/LiveViewer"


PROJECT_ROOT = find_project_root()
ATTR_JSON = os.path.join(PROJECT_ROOT, "v1112_compare", "attr.json")
LOG_DIR = os.path.join(PROJECT_ROOT, "v1112_compare", "logs")


def load_items(attr_path=None):
    """把 attr.json 展开成待评条目，返回 (items, problems)。

    attr.json 形如 [{"id": 片段号, "ours": 路径, "avtr-1": 路径, ...}]，
    除 id 外的每个键都是一个独立方法。路径不存在的条目不进评测队列，
    而是收集进 problems 由界面提示——静默跳过会让"共 N 段"对不上账。
    """
    attr_path = attr_path or ATTR_JSON
    with open(attr_path, "r", encoding="utf-8") as f:
        records = json.load(f)
    items, problems = [], []
    for rec in records:
        clip_id = rec.get("id")
        for method, raw in rec.items():
            if method == "id":
                continue
            path = os.path.abspath(raw)
            if not os.path.isfile(path):
                problems.append(f"片段 {clip_id} / {method}: 文件不存在 {raw}")
                continue
            items.append({"clip_id": clip_id, "method": method, "path": path})
    return items, problems


def method_order(items):
    """按 attr.json 中方法首次出现的顺序固定列序，汇总表的列才不会乱跳。"""
    seen = []
    for it in items:
        if it["method"] not in seen:
            seen.append(it["method"])
    return seen


def build_groups(items, rng):
    """按片段分组（同一段内容的所有方法在一组），随机打乱组序与组内位置。

    组内打乱用 slots 保存 items 下标的新顺序，渲染和落盘都读这一份，
    因此"显示顺序"和"CSV 顺序"永远一致，不可能错位。
    """
    by_clip = OrderedDict()
    for idx, it in enumerate(items):
        by_clip.setdefault(it["clip_id"], []).append(idx)
    groups = [{"clip_id": c, "slots": list(idxs)} for c, idxs in by_clip.items()]
    rng.shuffle(groups)
    for g in groups:
        rng.shuffle(g["slots"])
    return groups


def slot_label(pos: int) -> str:
    """组内位置 -> 视频 A/B/C/D（超过 8 个就退回数字）。"""
    return "ABCDEFGHIJ"[pos] if pos < 10 else str(pos + 1)


_VIDEO_URI_CACHE = {}


def video_uri(path: str) -> str:
    """base64 data URI，按路径缓存。

    一组 5 个视频来回翻看会反复渲染同一批文件，缓存后切组是纯字符串拼接；
    25 个 704x704 片段全缓存约 30MB，相对省掉的重复读盘+编码很划算。
    """
    if path not in _VIDEO_URI_CACHE:
        mime, _ = mimetypes.guess_type(path)
        with open(path, "rb") as f:
            _VIDEO_URI_CACHE[path] = (
                f"data:{mime or 'video/mp4'};base64,"
                + base64.b64encode(f.read()).decode("ascii")
            )
    return _VIDEO_URI_CACHE[path]


def video_html(path: str, max_width: int = VIDEO_MAX_WIDTH) -> str:
    """内嵌播放，宽高比锁定 1:1，带 us-video 类名供"全部播放"定位。

    素材本身是 704x704 正方形，锁定 aspect-ratio 后 element 与内容同比例，
    object-fit:cover 因此不会裁切也不会出现黑边——旧版把视频塞进满宽黑底容器
    再限制 max-height，容器宽度远大于 480px，左右两侧就空出了一大片黑边。

    页面上任何地方都不出现文件名或目录名，看 HTML 源码也翻不出方法归属。
    """
    return (
        '<div style="width:100%;display:flex;justify-content:center;">'
        f'<video class="us-video" controls preload="auto" playsinline '
        f'style="display:block;width:100%;max-width:{max_width}px;aspect-ratio:1/1;'
        f'object-fit:cover;border-radius:4px;background:#111;" '
        f'src="{video_uri(path)}"></video>'
        '</div>'
    )


# 「全部播放」用页面级 JS 同时驱动当前组的全部播放器：比较"反应时机"时
# 各视频必须从同一时刻起播，否则先后差半秒就分不出是模型问题还是观看顺序。
PLAY_ALL_JS = """
(function(){
  var vids = document.querySelectorAll('video.us-video');
  vids.forEach(function(v){ try { v.loop = __LOOP__; v.currentTime = 0; v.play(); }
                            catch (e) {} });
})();
"""

PAUSE_ALL_JS = """
document.querySelectorAll('video.us-video').forEach(function(v){ v.pause(); });
"""


def fmt_score(x: float) -> str:
    return f"{x:.2f}"


def variant_scores(scores: dict, idx: int) -> dict:
    """取某个视频已填的分数，缺维度就是缺，不做 0 分填充。"""
    return dict(scores.get(idx) or {})


def variant_complete(scores: dict, idx: int) -> bool:
    rec = scores.get(idx) or {}
    return all(isinstance(rec.get(k), int) for k in DIM_KEYS)


def group_complete(scores: dict, group: dict) -> bool:
    return all(variant_complete(scores, idx) for idx in group["slots"])


def count_variants_done(scores, groups):
    return sum(1 for g in groups for idx in g["slots"] if variant_complete(scores, idx))


def validate_log_dir(path: str):
    """确保 logs/ 存在。第一次评测时目录可能还没建。"""
    try:
        os.makedirs(path, exist_ok=True)
    except OSError as e:
        return f"无法创建日志目录 {path}: {e}"
    return ""


def save_results(items, groups, scores, rated_at, subject, seed, stamp, log_dir=None):
    """按 <YYYYMMDD_HHMMSS>.csv 落盘，一行一个视频，按展示顺序排列。

    同一秒内重复提交会撞名，那时退化成 <stamp>-2.csv；正常一次评测不会碰到。
    返回 (实际路径, 去掉 .csv 的基名，用作 session_id)。
    """
    log_dir = log_dir or LOG_DIR
    err = validate_log_dir(log_dir)
    if err:
        raise OSError(err)
    path = os.path.join(log_dir, f"{stamp}.csv")
    n = 2
    while os.path.exists(path):
        path = os.path.join(log_dir, f"{stamp}-{n}.csv")
        n += 1
    base = os.path.splitext(os.path.basename(path))[0]
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        writer.writeheader()
        order = 0
        for g_pos, g in enumerate(groups, 1):
            for s_pos, idx in enumerate(g["slots"]):
                it = items[idx]
                rec = variant_scores(scores, idx)
                nat, tim = rec.get("naturalness"), rec.get("timing")
                order += 1
                avg = ("%.4f" % ((nat + tim) / 2)) if (nat and tim) else ""
                writer.writerow({
                    "group_index": g_pos,
                    "slot": slot_label(s_pos),
                    "order_index": order,
                    "clip_id": it["clip_id"],
                    "method": it["method"],
                    "naturalness": nat if nat else "",
                    "timing": tim if tim else "",
                    "avg": avg,
                    "subject": subject,
                    "session_id": base,
                    "seed": seed,
                    "rated_at": rated_at.get(idx, ""),
                    "path": it["path"],
                })
    return path, base


def summarize(items, groups, scores, methods):
    """算方法级与 片段 × 方法 的均值，两个维度各一套，另附两维平均。

    评分键是 items 下标、与展示顺序无关，所以来回翻页改分不会算错。
    """
    by_method = defaultdict(lambda: {k: [] for k in DIM_KEYS})
    by_clip = defaultdict(lambda: defaultdict(lambda: {k: [] for k in DIM_KEYS}))
    for g in groups:
        for idx in g["slots"]:
            rec = variant_scores(scores, idx)
            if not rec:
                continue
            it = items[idx]
            for k in DIM_KEYS:
                if isinstance(rec.get(k), int):
                    by_method[it["method"]][k].append(float(rec[k]))
                    by_clip[it["clip_id"]][it["method"]][k].append(float(rec[k]))

    def stats(vals):
        if not vals:
            return None
        return {
            "n": len(vals),
            "mean": statistics.fmean(vals),
            "std": statistics.stdev(vals) if len(vals) > 1 else 0.0,
            "min": min(vals),
            "max": max(vals),
        }

    method_rows = {}
    for m in methods:
        dims = {k: stats(by_method[m][k]) for k in DIM_KEYS}
        pooled = by_method[m]["naturalness"] + by_method[m]["timing"]
        dims["avg"] = stats(pooled)
        method_rows[m] = dims
    clip_rows = {
        c: {m: {k: statistics.fmean(v) for k, v in d.items() if v}
            for m, d in mrows.items()}
        for c, mrows in by_clip.items()
    }
    return method_rows, clip_rows


def save_summary(method_rows, methods, session_id, log_dir=None):
    """方法级均值单独落一份，方便直接贴进实验记录或画图。"""
    log_dir = log_dir or LOG_DIR
    path = os.path.join(log_dir, f"{session_id}_summary.csv")
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["method", "n_naturalness", "naturalness_mean", "naturalness_std",
                         "n_timing", "timing_mean", "timing_std", "avg_mean"])
        for m in methods:
            dims = method_rows.get(m)
            if not dims or not dims["avg"]:
                continue
            nat, tim = dims.get("naturalness"), dims.get("timing")
            writer.writerow([
                m,
                nat["n"] if nat else 0,
                f'{nat["mean"]:.4f}' if nat else "",
                f'{nat["std"]:.4f}' if nat else "",
                tim["n"] if tim else 0,
                f'{tim["mean"]:.4f}' if tim else "",
                f'{tim["std"]:.4f}' if tim else "",
                f'{dims["avg"]["mean"]:.4f}',
            ])
    return path


def _table(headers, rows, td=None):
    th = 'style="border:1px solid #ddd;padding:4px 9px;background:#f5f5f5;"'
    td = td or 'style="border:1px solid #ddd;padding:4px 9px;text-align:center;"'
    out = ['<table style="border-collapse:collapse;margin-top:6px;font-size:12px;">',
           "<tr>" + "".join(f"<th {th}>{h}</th>" for h in headers) + "</tr>"]
    for r in rows:
        out.append("<tr>" + "".join(f"<td {td}>{c}</td>" for c in r) + "</tr>")
    out.append("</table>")
    return "".join(out)


def summary_html(method_rows, clip_rows, methods, mapping_rows):
    """交卷后展示：方法级双维度均值、各片段均分、以及视频字母 ↔ 方法 的对照。"""
    out = ['<div style="margin-top:16px;">',
           '<b>方法平均分（MOS 1–5）</b>']

    rows = []
    ranked = sorted(
        [(m, d) for m, d in method_rows.items() if d.get("avg")],
        key=lambda kv: -kv[1]["avg"]["mean"],
    )
    for m, d in ranked:
        nat, tim, avg = d.get("naturalness"), d.get("timing"), d["avg"]
        rows.append([
            f"<b>{m}</b>",
            nat["n"] if nat else 0,
            fmt_score(nat["mean"]) if nat else "–",
            f'{nat["std"]:.2f}' if nat else "–",
            fmt_score(tim["mean"]) if tim else "–",
            f'{tim["std"]:.2f}' if tim else "–",
            f"<b>{fmt_score(avg['mean'])}</b>",
        ])
    out.append(_table(["方法", "n", "自然程度", "±std", "时机恰当", "±std", "两维平均"], rows))

    for key in DIM_KEYS:
        out.append(f'<b style="display:inline-block;margin-top:14px;">各片段均分 · {DIM_NAMES[key]}</b>')
        rows = []
        for clip, mrow in clip_rows.items():
            rows.append([clip] + [
                fmt_score(mrow[m][key]) if m in mrow and key in mrow[m] else "–"
                for m in methods
            ])
        out.append(_table(["片段"] + list(methods), rows))

    if mapping_rows:
        out.append('<b style="display:inline-block;margin-top:14px;">'
                   '本轮位置对照（同组内随机）</b>')
        out.append(_table(["组", "位置", "方法"], mapping_rows))
    out.append("</div>")
    return "".join(out)


def load_all_logs(log_dir=None):
    """汇总 logs/ 下所有会话 CSV（跳过 *_summary.csv），供多人评测后统一统计。

    兼容只写了一列 score 的旧文件：那种情况下两个维度都取同一个值。
    subject 留空的会话按 session_id 归类，不填评价者也不会全被并成一堆。
    """
    log_dir = log_dir or LOG_DIR
    rows = []
    for p in sorted(glob.glob(os.path.join(log_dir, "*.csv"))):
        if p.endswith("_summary.csv"):
            continue
        try:
            with open(p, "r", encoding="utf-8-sig", newline="") as f:
                for r in csv.DictReader(f):
                    for key in DIM_KEYS:
                        raw = (r.get(key) or "").strip()
                        if not raw and key == DIM_KEYS[0]:
                            raw = (r.get("score") or "").strip()   # 旧格式兜底
                        if not raw:
                            continue
                        try:
                            val = float(raw)
                        except ValueError:
                            continue
                        rows.append({
                            "file": os.path.basename(p),
                            "subject": (r.get("subject") or "").strip() or r.get("session_id", ""),
                            "clip_id": r.get("clip_id", ""),
                            "method": r.get("method", ""),
                            "dimension": key,
                            "score": val,
                        })
        except (OSError, ValueError, KeyError):
            continue   # 单个坏文件不连坐其它会话
    return rows


In [ ]:
subject_input = widgets.Text(
    value="",
    description="评价者",
    placeholder="姓名或编号，只写进 CSV（可留空）",
    layout=widgets.Layout(width="430px"),
    style={"description_width": "70px"},
)
auto_cb = widgets.Checkbox(
    value=AUTO_ADVANCE_DEFAULT,
    description="本组评完自动下一组",
    indent=False,
    layout=widgets.Layout(width="200px"),
)

progress = widgets.IntProgress(
    min=0, max=1, value=0, bar_style="info",
    layout=widgets.Layout(width="100%"),
)
progress_label = widgets.HTML()
group_label = widgets.HTML(layout=widgets.Layout(width="100%", margin="4px 0 2px 0"))
stage_area = widgets.VBox([], layout=widgets.Layout(width="100%"))

prev_btn = widgets.Button(description="上一组", icon="step-backward",
                          layout=widgets.Layout(margin="0 4px"))
next_btn = widgets.Button(description="下一组", icon="step-forward",
                          layout=widgets.Layout(margin="0 4px"))
jump_btn = widgets.Button(description="跳到未完成", icon="crosshairs",
                          layout=widgets.Layout(margin="0 4px"))
play_all_btn = widgets.Button(description="全部播放", icon="play",
                              button_style="info", layout=widgets.Layout(margin="0 4px"))
pause_all_btn = widgets.Button(description="全部暂停", icon="pause",
                               layout=widgets.Layout(margin="0 4px"))
loop_cb = widgets.Checkbox(
    value=True, description="循环播放", indent=False,
    layout=widgets.Layout(width="110px"),
)
restart_btn = widgets.Button(description="重新开始（重新打乱）", icon="refresh",
                             layout=widgets.Layout(margin="0 4px"))
submit_btn = widgets.Button(description="提交并保存 CSV", button_style="primary",
                            icon="save", layout=widgets.Layout(margin="0 4px"))

status_out = widgets.Output()
js_out = widgets.Output()      # 只用来发 Javascript，不展示任何内容
summary_area = widgets.HTML()

# 可变状态集中在一个 dict（同 video_compare.ipynb 的写法）。
# groups 已在内存里定好展示顺序；scores 的键是 items 下标而不是展示位置，
# 于是翻页、改分、重新打乱三者互不干扰。syncing 用于区分"人点的"和"程序回填的"。
state = {
    "items": [],
    "methods": [],
    "groups": [],
    "pos": 0,
    "scores": {},        # items 下标 -> {"naturalness": int, "timing": int}
    "rated_at": {},      # items 下标 -> ISO 时间
    "seed": None,
    "session_id": None,  # 落盘后才知道
    "saved": False,
    "confirm_restart": False,
    "syncing": False,
    "rendered_pos": None,   # 当前 stage_area 里放的是第几组，避免无谓重建
    "cells": [],            # [{"idx":.., "toggles": {dim: ToggleButtons}}]
}


def current_group():
    if not state["groups"]:
        return None
    return state["groups"][state["pos"]]


def make_observer(idx, dim):
    """观察 ToggleButtons 的 value 变化，把分数写回 state。

    用 observe 而不是 on_click：ToggleButtons 没有 on_click，
    且 observe 能同时接住"点按钮"和"程序回填"两种情况，后者用 syncing 过滤掉。
    """
    def _obs(change):
        if state["syncing"] or state["saved"]:
            return
        group = current_group()
        if group is None:
            return
        was_complete = group_complete(state["scores"], group)

        val = change["new"]
        rec = state["scores"].setdefault(idx, {})
        if val is None:
            rec.pop(dim, None)
        else:
            rec[dim] = int(val)
        if not rec:
            state["scores"].pop(idx, None)
        state["rated_at"][idx] = datetime.datetime.now().isoformat(timespec="seconds")

        refresh_progress()
        if (auto_cb.value and not was_complete
                and group_complete(state["scores"], group)
                and state["pos"] < len(state["groups"]) - 1):
            # 只在"本组由未完成变为完成"的瞬间自动前进，事后回来改分不会被弹走
            move_to(state["pos"] + 1)
    return _obs


def build_cell(idx, slot_pos):
    """一个视频格子：正方形播放器 + 位置标签 + 两个维度的分数按钮。"""
    label = widgets.HTML(
        f'<div style="text-align:center;font-size:13px;font-weight:600;'
        f'margin:2px 0 4px 0;">视频 {slot_label(slot_pos)}</div>'
    )
    video = widgets.HTML(video_html(state["items"][idx]["path"]))
    toggles = {}
    children = [label, video]
    for key, name, tip in DIMENSIONS:
        tb = widgets.ToggleButtons(
            options=[(str(s), s) for s in range(SCORE_MIN, SCORE_MAX + 1)],
            value=None,
            description=name,
            tooltips=[tip] * (SCORE_MAX - SCORE_MIN + 1),
            style={"description_width": "60px", "button_width": "28px"},
            layout=widgets.Layout(width="100%"),
        )
        tb.observe(make_observer(idx, key), names="value")
        toggles[key] = tb
        children.append(tb)
    cell = widgets.VBox(
        children,
        layout=widgets.Layout(flex="1 1 0", min_width="0", padding="0 6px",
                              align_items="stretch"),
    )
    return cell, toggles


def build_group_rows(group):
    """把当前组的视频按 GROUP_COLUMNS 分块换行，返回 (rows, cells)。

    一组 5 个正好一行；万一某个片段的方法数超过 GROUP_COLUMNS 也只会多一行，不会挤成一条。
    """
    cells, rows, chunk = [], [], []
    for s_pos, idx in enumerate(group["slots"]):
        cell, toggles = build_cell(idx, s_pos)
        cells.append({"idx": idx, "toggles": toggles})
        chunk.append(cell)
        if len(chunk) == GROUP_COLUMNS:
            rows.append(widgets.HBox(chunk, layout=widgets.Layout(
                width="100%", justify_content="center", align_items="flex-start")))
            chunk = []
    if chunk:
        rows.append(widgets.HBox(chunk, layout=widgets.Layout(
            width="100%", justify_content="center", align_items="flex-start")))
    return rows, cells


def sync_scores():
    """把 state 里的分数回填到当前组的按钮上。只改 value，不重建 widget。

    重建会让用户正在看的视频重新加载并跳回 0 秒，所以打分只刷新按钮和进度条。
    """
    state["syncing"] = True
    try:
        for cell in state["cells"]:
            rec = variant_scores(state["scores"], cell["idx"])
            for key, tb in cell["toggles"].items():
                tb.value = rec.get(key)
                tb.disabled = state["saved"]
    finally:
        state["syncing"] = False


def refresh_progress():
    groups = state["groups"]
    total_groups = len(groups)
    total_variants = sum(len(g["slots"]) for g in groups)
    done_variants = count_variants_done(state["scores"], groups)
    done_groups = sum(1 for g in groups if group_complete(state["scores"], g))

    progress.max = total_variants or 1
    progress.value = done_variants
    progress.bar_style = "success" if done_variants == total_variants else "info"

    group = current_group()
    here = ""
    if group:
        n = len(group["slots"])
        ready = sum(1 for i in group["slots"] if variant_complete(state["scores"], i))
        here = f'　<span style="color:#666;">本组 {ready} / {n}</span>'
    progress_label.value = (
        f'<span style="font-size:13px;">已完成 <b>{done_groups}</b> / {total_groups} 组'
        f'（{done_variants} / {total_variants} 个视频）{here}</span>'
    )
    prev_btn.disabled = state["pos"] == 0 or state["saved"]
    next_btn.disabled = state["pos"] >= total_groups - 1 or state["saved"]


def render(force=False):
    """切组时重建播放器与按钮，同组内只同步分数。"""
    group = current_group()
    if group is None:
        stage_area.children = ()
        group_label.value = ""
        state["cells"] = []
        refresh_progress()
        return
    group_label.value = (
        f'<div style="font-size:13px;color:#333;">第 <b>{state["pos"] + 1}</b> / '
        f'{len(state["groups"])} 组 · 组内位置每轮随机，请逐个独立打分</div>'
    )
    if force or state["rendered_pos"] != state["pos"]:
        rows, cells = build_group_rows(group)
        stage_area.children = tuple(rows)
        state["cells"] = cells
        state["rendered_pos"] = state["pos"]
    else:
        sync_scores()
    refresh_progress()


def move_to(pos):
    if not state["groups"]:
        return
    state["pos"] = max(0, min(pos, len(state["groups"]) - 1))
    render()


def on_prev(_):
    move_to(state["pos"] - 1)


def on_next(_):
    move_to(state["pos"] + 1)


def run_js(code: str):
    """把一段 JS 塞进隐藏 Output 里执行。

    用 Javascript 而不是把 onclick 写进 HTML：小组件模式下 HTML 里的事件属性
    会被框架过滤，而 display(Javascript) 走的是标准内核通道，JupyterLab / Voilà 都认。
    """
    with js_out:
        clear_output(wait=True)
        display(Javascript(code))


def on_play_all(_):
    run_js(PLAY_ALL_JS.replace("__LOOP__", "true" if loop_cb.value else "false"))


def on_pause_all(_):
    run_js(PAUSE_ALL_JS)


def on_jump(_):
    """跳到第一个还没评完的组；都评完了就提示可以提交。"""
    with status_out:
        clear_output()
    for pos, g in enumerate(state["groups"]):
        if not group_complete(state["scores"], g):
            move_to(pos)
            return
    with status_out:
        display(HTML('<div style="color:#1a7f37;">全部组都已评完，可以提交了。</div>'))


def start_session(_=None):
    """读 attr.json → 分组 → 打乱 → 清空分数。只有这一条路能开新的一轮。"""
    with status_out:
        clear_output()
    items, problems = load_items()
    if not items:
        with status_out:
            display(HTML('<div style="color:#c00;">没有读到任何可播放的视频，请检查 '
                         f'<code>{ATTR_JSON}</code></div>'))
        return

    seed = random.randrange(1, 2 ** 31)
    groups = build_groups(items, random.Random(seed))
    state.update({
        "items": items,
        "methods": method_order(items),
        "groups": groups,
        "pos": 0,
        "scores": {},
        "rated_at": {},
        "seed": seed,
        "session_id": None,
        "saved": False,
        "confirm_restart": False,
        "rendered_pos": None,
        "cells": [],
    })
    restart_btn.description = "重新开始（重新打乱）"
    restart_btn.button_style = ""
    submit_btn.disabled = False
    summary_area.value = ""
    render(force=True)

    n_variants = len(items)
    notes = [f"共 {len(groups)} 组（每组 {len(groups[0]['slots'])} 个视频）"
             f"= {n_variants} 个视频待评，组序与组内位置均已随机（seed={seed}）"]
    if problems:
        notes.append(f"以下 {len(problems)} 条被跳过：\n  " + "\n  ".join(problems))
    color = "orange" if problems else "green"
    with status_out:
        display(HTML(f'<div style="color:{color};white-space:pre-wrap;">'
                     + "\n".join(notes) + "</div>"))


def on_restart(_):
    """有未保存的分数时要求点两次——评到一半误触清空太伤人。"""
    done = len(state["scores"])
    if done and not state["saved"] and not state["confirm_restart"]:
        state["confirm_restart"] = True
        restart_btn.description = f"再点一次确认丢弃 {done} 条评分"
        restart_btn.button_style = "danger"
        return
    start_session()


def on_submit(_):
    with status_out:
        clear_output()
    if not state["groups"]:
        return
    if state["saved"]:
        with status_out:
            display(HTML('<div style="color:#c60;">本轮已保存过；要再评一轮请点「重新开始」。</div>'))
        return

    # 漏评不静默补 0：先跳到第一个没评完的组，让人补完再提交
    incomplete = [pos + 1 for pos, g in enumerate(state["groups"])
                  if not group_complete(state["scores"], g)]
    if incomplete:
        move_to(incomplete[0] - 1)
        shown = ", ".join(map(str, incomplete[:20])) + (" …" if len(incomplete) > 20 else "")
        with status_out:
            display(HTML(f'<div style="color:#c60;">还有 {len(incomplete)} 组没评完'
                         f'（第 {shown} 组），已跳到第一处未完成。</div>'))
        return

    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    try:
        path, base = save_results(
            state["items"], state["groups"], state["scores"], state["rated_at"],
            subject_input.value.strip(), state["seed"], stamp,
        )
    except OSError as e:
        with status_out:
            display(HTML(f'<div style="color:#c00;">写文件失败：{e}</div>'))
        return

    state["session_id"] = base
    state["saved"] = True
    submit_btn.disabled = True
    sync_scores()
    refresh_progress()

    method_rows, clip_rows = summarize(state["items"], state["groups"],
                                      state["scores"], state["methods"])
    summary_path = ""
    if WRITE_SUMMARY_CSV:
        try:
            summary_path = save_summary(method_rows, state["methods"], base)
        except OSError as e:
            summary_path = f"（汇总文件写入失败：{e}）"

    mapping_rows = []
    for g_pos, g in enumerate(state["groups"], 1):
        for s_pos, idx in enumerate(g["slots"]):
            mapping_rows.append([g_pos, slot_label(s_pos), state["items"][idx]["method"]])

    extra = ""
    if summary_path and not summary_path.startswith("（"):
        extra = f"<br>方法均值：<code>{summary_path}</code>"
    elif summary_path:
        extra = f"<br><span style='color:#c60;'>{summary_path}</span>"
    n_rows = sum(len(g["slots"]) for g in state["groups"])
    with status_out:
        display(HTML(f'<div style="color:#1a7f37;">已保存 {n_rows} 条评分：'
                     f'<code>{path}</code>{extra}</div>'))
    if SHOW_SUMMARY_ON_SUBMIT:
        summary_area.value = summary_html(method_rows, clip_rows, state["methods"],
                                          mapping_rows)


prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
jump_btn.on_click(on_jump)
play_all_btn.on_click(on_play_all)
pause_all_btn.on_click(on_pause_all)
restart_btn.on_click(on_restart)
submit_btn.on_click(on_submit)

help_html = widgets.HTML(
    "<div style='color:#666;font-size:12px;margin:4px 0;'>"
    "同一组是<b>同一段内容的不同方法</b>，位置（A/B/C/D）每轮随机，界面不显示方法名。"
    "每个视频打两项：<b>自然程度</b>（有没有崩坏、伪影、口型不自然）和"
    "<b>时机恰当</b>（反应时机是否合理、是否同步）。<br>"
    "1 很差 / 2 较差 / 3 一般 / 4 较好 / 5 很好；点其它分值即可覆盖。"
    "「提交并保存 CSV」会检查漏评并跳到第一个没评完的组。<br>"
    "评<b>时机恰当</b>时建议先点<b>全部播放</b>，让同组所有视频从同一时刻起播，"
    "避免先后差半秒被误判成模型反应延迟。<br>"
    "部分素材可能没有音轨（原视频常见），请照常打分，不要因此扣分。"
    "</div>"
)

ui = widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:2px;'>主观评测 · 反应自然程度 / 时机恰当</h3>"),
    widgets.HBox([subject_input, auto_cb],
                 layout=widgets.Layout(align_items="center", margin="6px 0")),
    progress,
    progress_label,
    group_label,
    widgets.HBox([play_all_btn, pause_all_btn, loop_cb],
                 layout=widgets.Layout(justify_content="center",
                                       align_items="center", margin="4px 0")),
    stage_area,
    widgets.HBox([prev_btn, next_btn, jump_btn, restart_btn, submit_btn],
                 layout=widgets.Layout(justify_content="center", margin="10px 0 6px 0")),
    help_html,
    status_out,
    js_out,
    summary_area,
], layout=widgets.Layout(width="100%"))

display(ui)
start_session()


In [ ]:
# ===== 汇总统计：读 logs/ 下所有会话（多人评测结束后跑这一格）=====
rows = load_all_logs()
if not rows:
    print(f"{LOG_DIR} 下还没有评分记录")
else:
    pooled = defaultdict(lambda: {"naturalness": [], "timing": []})
    for r in rows:
        pooled[r["method"]][r["dimension"]].append(r["score"])

    sessions = sorted({r["file"] for r in rows})
    subjects = sorted({r["subject"] for r in rows if r["subject"]})
    print(f"会话 {len(sessions)} 个，评分 {len(rows)} 条；"
          f"评价者：{', '.join(subjects) or '(未填)'}")

    ranked = sorted(
        pooled.items(),
        key=lambda kv: -statistics.fmean(kv[1]["naturalness"] + kv[1]["timing"]),
    )
    td = 'style="border:1px solid #ddd;padding:4px 10px;text-align:center;"'
    parts = ['<table style="border-collapse:collapse;font-size:13px;">',
             f"<tr><th {td}>名次</th><th {td}>方法</th><th {td}>n</th>"
             f"<th {td}>自然程度</th><th {td}>时机恰当</th><th {td}>两维平均</th></tr>"]
    for rank, (m, d) in enumerate(ranked, 1):
        nat, tim = d["naturalness"], d["timing"]
        both = nat + tim
        parts.append(
            f"<tr><td {td}>{rank}</td><td {td}>{m}</td><td {td}>{len(both)}</td>"
            f"<td {td}>{statistics.fmean(nat):.2f}</td>"
            f"<td {td}>{statistics.fmean(tim):.2f}</td>"
            f"<td {td}><b>{statistics.fmean(both):.2f}</b></td></tr>"
        )
    parts.append("</table>")
    display(HTML("".join(parts)))
